# Phase 5 - which upgrades make the ESP32 counter better? (all 5 datasets, leave one site out)

Starting point = the Phase 4 final model: LightGBM regression on the 10 base cues, trained on
WiCount + mat80 + WiMANS (+ CSI-Bench where usable), `empty_cal` = 20 s empty room at install.

| Test | What changes | Needs on the device |
|---|---|---|
| `extra_reg` | + 9 "separate movers" cues | 9 more cues in the C code |
| `base_clf4` | predicts the 0 / 1 / 2 / 3+ class directly (4-class LightGBM) | multi-class tree export |
| `mem_reg` | + 30 s memory: mean / std of the last 12 windows' motion cues (streams only) | a 12-window buffer |
| two-point calibration | install = 20 s empty room **+ 20 s with 2 people**; estimates are stretched so empty -> 0 and 2 -> 2 | one extra install step |
| 2 / 3 links | 2 or 3 receivers, each with its own counter + calibration, estimates averaged | more ESP32 boards |

**Datasets (all 5):** mat80 (router 80 MHz, 0-10), WiCount (ESP32-S3, 0-3), WiMANS 5 GHz (0-5), CSI-Bench
Localization (1-2), CountFi (1-10, never trained on). Every site is scored by a model that never saw it
(mat80 setup, WiMANS / CSI-Bench room, WiCount file, CountFi = whole dataset).

**Links per dataset** (what "2 receivers" means here): mat80 = different antennas of one router (same sub-band),
WiMANS = different receive antennas, **CSI-Bench = different devices**, **CountFi = the two receivers**,
WiCount = one link only. Antennas of one device sit close together, so mat80 / WiMANS give a lower bound
for the gain of two ESP32 boards on different walls.

Output: `/kaggle/working/phase5/` (`PHASE5_RESULTS.md`, CSV tables, figures).
No new model is exported: every upgrade here needs a change to `esp32_counter.py` / the firmware first,
so this run decides which changes are worth making.

In [ ]:
%%writefile /kaggle/working/esp32_counter.py
"""ESP32-S3 people counter - reference implementation of the final model.

Everything the firmware has to do, written in Python, packet by packet:

    raw CSI amplitude (52 tones)
      -> per-packet gain normalisation (divide by the median over tones)
      -> 50 Hz bins (packets averaged per 20 ms), gaps up to 60 ms interpolated
      -> causal baseline per tone (EMA, 30 s) -> relative fluctuation z = x / baseline - 1
      -> 5 s windows every 2.5 s (first 10 s skipped while the baseline settles)
      -> 10 count features
      -> log of the magnitude features, minus the empty-room offset (if calibrated)
      -> tree model (LightGBM regression, exported to plain arrays / C header)
      -> clip to the trained count range -> median of the last 30 s -> people count

The batch functions (used for training) and StreamingCounter (what the device runs)
share the same code, and test_esp32_counter.py checks that they give the same numbers.
Nothing here looks at future packets.
"""

import json
import math
import os
import time
from collections import deque

import numpy as np
from scipy.signal import lfilter


# ===============================================================
# Settings (identical on the ESP32)
# ===============================================================
FS = 50                  # sample rate of the shared grid (Hz)
WIN_S = 5.0              # window length (s)
HOP_S = 2.5              # window hop (s)
WIN = int(WIN_S * FS)    # 250 bins
HOP = int(HOP_S * FS)    # 125 bins
N_TONES = 52

MAX_GAP_BINS = 3         # gaps up to 3 bins (60 ms) are interpolated
MAX_MISSING = 0.20       # windows with more missing bins than this are skipped

TAU_S = 30.0             # time constant of the causal per-tone baseline
ALPHA = 1.0 / (TAU_S * FS)
WARMUP_S = 10.0          # no output for the first 10 s (baseline settling)
WARMUP_BINS = int(WARMUP_S * FS)

CAL_S = 20.0             # empty-room calibration after the warm-up
N_CAL_WINDOWS = int((CAL_S - WIN_S) / HOP_S) + 1     # 7 windows

SMOOTH_S = 30.0          # output = median of the last 30 s of window estimates
SMOOTH_N = int(SMOOTH_S / HOP_S)                    # 12 windows

EPS = 1e-6

FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]

# magnitude features go through log() so the empty-room offset acts as a ratio
LOG_FEATURES = ["cv_median", "cv_p90", "cv_p10", "diff_energy", "mean_std"]
LOG_IDX = [FEATURE_NAMES.index(name) for name in LOG_FEATURES]


def settings_dict():
    return {"fs_hz": FS, "win_s": WIN_S, "hop_s": HOP_S, "n_tones": N_TONES,
            "max_gap_bins": MAX_GAP_BINS, "max_missing": MAX_MISSING,
            "tau_s": TAU_S, "warmup_s": WARMUP_S, "cal_s": CAL_S,
            "n_cal_windows": N_CAL_WINDOWS, "smooth_s": SMOOTH_S, "smooth_n": SMOOTH_N,
            "feature_names": FEATURE_NAMES, "log_features": LOG_FEATURES}


# ===============================================================
# Batch preprocessing (training)
# ===============================================================
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + EPS)


def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def causal_normalise(X, valid):
    """Each tone relative to its own past (EMA baseline): z = x / baseline - 1.

    The baseline only moves on valid bins and starts at the first valid value.
    Because it is linear, any constant per-tone scale of X cancels out, so the
    Phase 1 series (already divided by the recording mean) give the same z."""
    Z = np.full(X.shape, np.nan, dtype=np.float32)
    rows = np.flatnonzero(valid)
    if len(rows) == 0:
        return Z

    xs = X[rows].astype(np.float64)
    zi = (1.0 - ALPHA) * xs[:1]                             # first output = first value
    baseline, _ = lfilter([ALPHA], [1.0, -(1.0 - ALPHA)], xs, axis=0, zi=zi)

    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(baseline > EPS, xs / baseline - 1.0, 0.0)
    Z[rows] = z
    return Z


def window_starts(valid):
    """Window start bins: every HOP, after the warm-up, with enough valid bins."""
    n = len(valid)
    return [a for a in range(0, n - WIN + 1, HOP)
            if a >= WARMUP_BINS and valid[a:a + WIN].mean() >= 1 - MAX_MISSING]


# ===============================================================
# Count features (same code as Phase 1)
# ===============================================================
def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


def features_from_series(X, valid):
    """X: (bins, 52) amplitude on the 50 Hz grid (any per-tone scale).
    Returns window start bins and features (windows, 10)."""
    Z = causal_normalise(X, valid)
    starts = window_starts(valid)
    F = np.zeros((len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for j, a in enumerate(starts):
        F[j] = window_features(Z[a:a + WIN])
    return np.array(starts, dtype=np.int32), F


def features_from_raw(t, A):
    """Raw packets (times s, amplitude (packets, 52)) -> window starts + features."""
    order = np.argsort(t, kind="stable")
    t, A = np.asarray(t, dtype=np.float64)[order], np.asarray(A)[order]
    X, valid = resample_to_grid(t, gain_normalise(A.astype(np.float32)))
    X, valid = fill_short_gaps(X, valid)
    return features_from_series(X, valid)


# ===============================================================
# Model input, calibration, smoothing
# ===============================================================
def prepare_features(F, offset=None):
    """Raw features -> model input: NaN -> 0, log of magnitude features, minus offset."""
    T = np.nan_to_num(np.asarray(F, dtype=np.float64), nan=0.0)
    T = np.atleast_2d(T).copy()
    T[:, LOG_IDX] = np.log(np.maximum(T[:, LOG_IDX], 0.0) + EPS)
    if offset is not None:
        T = T - np.asarray(offset, dtype=np.float64)
    return T.astype(np.float32)


def calibration_offset(F_cal):
    """Empty-room offset = mean model input over the calibration windows."""
    return prepare_features(F_cal).astype(np.float64).mean(axis=0)


def smooth_predictions(pred):
    """Causal rolling median over the last SMOOTH_N window estimates."""
    pred = np.asarray(pred, dtype=np.float64)
    out = np.empty_like(pred)
    for i in range(len(pred)):
        out[i] = np.median(pred[max(0, i - SMOOTH_N + 1):i + 1])
    return out


def to_count(pred, max_count):
    return np.clip(np.rint(pred), 0, max_count).astype(int)


# ===============================================================
# Tree model: LightGBM -> flat arrays (Python, JSON, C header)
# ===============================================================
def _float32_at_most(value):
    """Largest float32 <= value, so 'x <= thr' in float32 equals 'x <= value' in double."""
    thr = np.float32(value)
    if float(thr) > value:
        thr = np.nextafter(thr, np.float32(-np.inf))
    return float(thr)


class TreeModel:
    """Sum of regression trees stored as flat node arrays.

    Node i: feature[i] = -1 for a leaf (value[i] = leaf output), otherwise go to
    left[i] if x[feature[i]] <= value[i] (threshold) else right[i]."""

    def __init__(self, roots, feature, value, left, right, max_count, n_features):
        self.roots = np.asarray(roots, dtype=np.int32)
        self.feature = np.asarray(feature, dtype=np.int32)
        self.value = np.asarray(value, dtype=np.float32)
        self.left = np.asarray(left, dtype=np.int32)
        self.right = np.asarray(right, dtype=np.int32)
        self.max_count = int(max_count)
        self.n_features = int(n_features)

    # ---------------------------------------------------------------
    # Build / save / load
    # ---------------------------------------------------------------
    @classmethod
    def from_lightgbm(cls, booster, max_count):
        dump = booster.dump_model()
        feature, value, left, right, roots = [], [], [], [], []

        def add(node):
            i = len(feature)
            feature.append(-1)
            value.append(0.0)
            left.append(-1)
            right.append(-1)

            if "leaf_value" in node:
                value[i] = float(np.float32(node["leaf_value"]))
                return i

            if node["decision_type"] != "<=":
                raise ValueError(f"unsupported split type {node['decision_type']}")
            if node.get("missing_type", "None") not in ("None", "NaN"):
                raise ValueError(f"unsupported missing type {node['missing_type']}")

            feature[i] = int(node["split_feature"])
            value[i] = _float32_at_most(float(node["threshold"]))
            left[i] = add(node["left_child"])
            right[i] = add(node["right_child"])
            return i

        for tree in dump["tree_info"]:
            roots.append(add(tree["tree_structure"]))

        return cls(roots, feature, value, left, right, max_count,
                   n_features=dump["max_feature_idx"] + 1)

    def to_dict(self):
        return {"roots": self.roots.tolist(), "feature": self.feature.tolist(),
                "value": [float(v) for v in self.value], "left": self.left.tolist(),
                "right": self.right.tolist(), "max_count": self.max_count,
                "n_features": self.n_features}

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.to_dict(), f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            d = json.load(f)
        return cls(d["roots"], d["feature"], d["value"], d["left"], d["right"],
                   d["max_count"], d["n_features"])

    # ---------------------------------------------------------------
    # Inference (float32, same order of operations as the C code)
    # ---------------------------------------------------------------
    def predict(self, X):
        X = np.atleast_2d(np.asarray(X, dtype=np.float32))
        if np.isnan(X).any():
            raise ValueError("model input contains NaN - use prepare_features()")

        rows = np.arange(len(X))
        total = np.zeros(len(X), dtype=np.float32)
        for root in self.roots:
            node = np.full(len(X), root, dtype=np.int32)
            while True:
                f = self.feature[node]
                split = f >= 0
                if not split.any():
                    break
                go_left = X[rows, np.maximum(f, 0)] <= self.value[node]
                node = np.where(split, np.where(go_left, self.left[node], self.right[node]), node)
            total += self.value[node]
        return total

    @property
    def n_nodes(self):
        return len(self.feature)

    # ---------------------------------------------------------------
    # C header for the ESP32 firmware
    # ---------------------------------------------------------------
    def write_c_header(self, path, note=""):
        idx_type = "int16_t" if self.n_nodes < 32767 else "int32_t"

        def array(ctype, name, values, fmt):
            body = ", ".join(fmt(v) for v in values)
            lines, line = [], "    "
            for item in body.split(", "):
                if len(line) + len(item) > 96:
                    lines.append(line.rstrip())
                    line = "    "
                line += item + ", "
            lines.append(line.rstrip().rstrip(","))
            return f"static const {ctype} {name}[{len(values)}] = {{\n" + "\n".join(lines) + "\n};\n"

        log_flags = [1 if name in LOG_FEATURES else 0 for name in FEATURE_NAMES]
        float_fmt = lambda v: f"{float(v):.9g}f"
        int_fmt = lambda v: str(int(v))

        text = [
            "/* ESP32-S3 people counter - generated by esp32_counter.py. Do not edit. */",
            f"/* {note} */" if note else "",
            "#ifndef CC_COUNTER_MODEL_H",
            "#define CC_COUNTER_MODEL_H",
            "",
            "#include <math.h>",
            "#include <stdint.h>",
            "",
            "/* ---- preprocessing settings (must match esp32_counter.py) ---- */",
            f"#define CC_FS_HZ          {FS}",
            f"#define CC_N_TONES        {N_TONES}",
            f"#define CC_WIN_BINS       {WIN}",
            f"#define CC_HOP_BINS       {HOP}",
            f"#define CC_MAX_GAP_BINS   {MAX_GAP_BINS}",
            f"#define CC_MAX_MISSING    {MAX_MISSING}f",
            f"#define CC_EMA_ALPHA      {ALPHA:.9g}f",
            f"#define CC_WARMUP_BINS    {WARMUP_BINS}",
            f"#define CC_N_CAL_WINDOWS  {N_CAL_WINDOWS}",
            f"#define CC_SMOOTH_N       {SMOOTH_N}",
            "",
            "/* ---- model ---- */",
            f"#define CC_N_FEATURES     {self.n_features}",
            f"#define CC_N_TREES        {len(self.roots)}",
            f"#define CC_N_NODES        {self.n_nodes}",
            f"#define CC_MAX_COUNT      {self.max_count}",
            "",
            "/* features, in order: " + ", ".join(FEATURE_NAMES) + " */",
            array("uint8_t", "cc_log_feature", log_flags, int_fmt),
            array(idx_type, "cc_roots", self.roots, int_fmt),
            array("int8_t", "cc_feature", self.feature, int_fmt),
            array("float", "cc_value", self.value, float_fmt),
            array(idx_type, "cc_left", self.left, int_fmt),
            array(idx_type, "cc_right", self.right, int_fmt),
            "/* raw window features -> model input. offset = NULL when not calibrated. */",
            "static inline void cc_prepare(const float *raw, const float *offset, float *out)",
            "{",
            "    for (int i = 0; i < CC_N_FEATURES; i++) {",
            "        float v = isnan(raw[i]) ? 0.0f : raw[i];",
            "        if (cc_log_feature[i]) {",
            "            v = logf(fmaxf(v, 0.0f) + 1e-6f);",
            "        }",
            "        if (offset) {",
            "            v -= offset[i];",
            "        }",
            "        out[i] = v;",
            "    }",
            "}",
            "",
            "/* model input -> people estimate (not yet clipped / smoothed) */",
            "static inline float cc_predict(const float *x)",
            "{",
            "    float total = 0.0f;",
            "    for (int t = 0; t < CC_N_TREES; t++) {",
            "        int n = cc_roots[t];",
            "        while (cc_feature[n] >= 0) {",
            "            n = (x[cc_feature[n]] <= cc_value[n]) ? cc_left[n] : cc_right[n];",
            "        }",
            "        total += cc_value[n];",
            "    }",
            "    return total;",
            "}",
            "",
            "#endif /* CC_COUNTER_MODEL_H */",
            "",
        ]
        with open(path, "w") as f:
            f.write("\n".join(text))
        return os.path.getsize(path)

    def flash_bytes(self):
        """Approximate flash used by the node arrays in the C header."""
        idx_bytes = 2 if self.n_nodes < 32767 else 4
        per_node = 1 + 4 + 2 * idx_bytes                    # feature + value + left + right
        return self.n_nodes * per_node + len(self.roots) * idx_bytes


# ===============================================================
# Streaming counter (what the ESP32 runs)
# ===============================================================
class StreamingCounter:
    """Feed packets one by one with push(t_seconds, amplitude_52).

    Every 2.5 s (after the warm-up) push() returns a result dict with the window
    features and, if a model is loaded, the raw / smoothed estimate and the count.
    Packets must arrive in time order; late packets are dropped."""

    def __init__(self, model=None, offset=None):
        self.model = model
        self.offset = None if offset is None else np.asarray(offset, dtype=np.float64)

        # 50 Hz binning
        self.t0 = None
        self.cur_bin = -1
        self.acc_sum = None
        self.acc_n = 0
        self.last_valid_bin = -1
        self.last_valid_x = None

        # causal baseline + window buffer
        self.baseline = None
        self.z_buf = deque(maxlen=WIN)
        self.v_buf = deque(maxlen=WIN)
        self.n_bins = 0

        # output smoothing + calibration
        self.recent = deque(maxlen=SMOOTH_N)
        self.cal_feats = None
        self.cal_target = 0

    # ---------------------------------------------------------------
    # Calibration: leave the room empty, call begin_calibration(), keep pushing
    # ---------------------------------------------------------------
    def begin_calibration(self, n_windows=N_CAL_WINDOWS):
        self.cal_feats = []
        self.cal_target = n_windows

    @property
    def calibrating(self):
        return self.cal_feats is not None

    # ---------------------------------------------------------------
    # Packet input
    # ---------------------------------------------------------------
    def push(self, t, amplitude):
        x = gain_normalise(np.asarray(amplitude, dtype=np.float32)[None])[0]
        if self.t0 is None:
            self.t0 = float(t)
        b = int(math.floor((float(t) - self.t0) * FS))

        results = []
        if b < self.cur_bin:
            return results                                  # late packet

        if b > self.cur_bin:
            if self.acc_n:
                results += self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.cur_bin = b
            self.acc_sum = np.zeros(N_TONES, dtype=np.float64)
            self.acc_n = 0

        self.acc_sum += x
        self.acc_n += 1
        return results

    def flush(self):
        """Close the last open bin (end of a recording)."""
        results = []
        if self.acc_n:
            results = self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.acc_n = 0
        return results

    # ---------------------------------------------------------------
    # Internals
    # ---------------------------------------------------------------
    def _close_bin(self, k, mean):
        x = mean.astype(np.float32)
        out = []

        if self.last_valid_bin >= 0:
            gap = k - self.last_valid_bin - 1
            if 0 < gap <= MAX_GAP_BINS:
                left = self.last_valid_bin
                for j in range(left + 1, k):
                    w = (j - left) / (k - left)
                    xi = (self.last_valid_x * (1 - w) + x * w).astype(np.float32)
                    out += self._add_bin(xi, True)
            else:
                for _ in range(gap):
                    out += self._add_bin(None, False)

        self.last_valid_bin, self.last_valid_x = k, x
        out += self._add_bin(x, True)
        return out

    def _add_bin(self, x, valid):
        if valid:
            xd = x.astype(np.float64)
            if self.baseline is None:
                self.baseline = xd.copy()
            else:
                self.baseline = self.baseline + ALPHA * (xd - self.baseline)
            with np.errstate(invalid="ignore", divide="ignore"):
                z = np.where(self.baseline > EPS, xd / self.baseline - 1.0, 0.0)
            z = z.astype(np.float32)
        else:
            z = np.full(N_TONES, np.nan, dtype=np.float32)

        self.z_buf.append(z)
        self.v_buf.append(valid)
        self.n_bins += 1

        a = self.n_bins - WIN
        if a < 0 or a % HOP or a < WARMUP_BINS:
            return []
        if np.mean(self.v_buf) < 1 - MAX_MISSING:
            return []

        feats = window_features(np.array(self.z_buf))
        return [self._decide(a, feats)]

    def _decide(self, start_bin, feats):
        res = {"t_start_s": start_bin / FS, "t_end_s": (start_bin + WIN) / FS,
               "features": feats}

        if self.calibrating:
            self.cal_feats.append(feats)
            res["status"] = "calibrating"
            if len(self.cal_feats) >= self.cal_target:
                self.offset = calibration_offset(np.array(self.cal_feats))
                self.cal_feats = None
                self.recent.clear()
                res["status"] = "calibrated"
            return res

        if self.model is None:
            return res

        x = prepare_features(feats[None], self.offset)
        raw = float(np.clip(self.model.predict(x)[0], 0, self.model.max_count))
        self.recent.append(raw)
        smoothed = float(np.median(self.recent))
        res.update(status="ok", raw=raw, smoothed=smoothed,
                   count=int(to_count(smoothed, self.model.max_count)))
        return res


# ===============================================================
# Model bundle (what goes to the device / the friend's demo)
# ===============================================================
def save_bundle(model_dir, tree_model, card, booster=None):
    """Writes trees.json, model_card.json, counter_model.h (+ model.txt if booster)."""
    os.makedirs(model_dir, exist_ok=True)
    tree_model.save(os.path.join(model_dir, "trees.json"))

    note = f"{card.get('name', 'counter')} | calibration={card.get('calibration')} | " \
           f"counts 0-{tree_model.max_count} | {time.strftime('%Y-%m-%d %H:%M')}"
    header_bytes = tree_model.write_c_header(os.path.join(model_dir, "counter_model.h"), note)

    if booster is not None:
        booster.save_model(os.path.join(model_dir, "model.txt"))

    card = {**card, "settings": settings_dict(), "n_trees": len(tree_model.roots),
            "n_nodes": tree_model.n_nodes, "flash_bytes_est": tree_model.flash_bytes(),
            "header_file_bytes": header_bytes}
    with open(os.path.join(model_dir, "model_card.json"), "w") as f:
        json.dump(card, f, indent=1)
    return card


def load_bundle(model_dir):
    """Returns (TreeModel, card). Use with StreamingCounter(model)."""
    model = TreeModel.load(os.path.join(model_dir, "trees.json"))
    with open(os.path.join(model_dir, "model_card.json")) as f:
        card = json.load(f)
    if card["settings"] != settings_dict():
        raise ValueError("model_card settings differ from this esp32_counter.py")
    return model, card

In [ ]:
# 1. Setup: load the Phase 5a cue tables
import glob
import itertools
import json
import os
import sys
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found - attach the output of the wifi-phase5-features kernel")
    return hits[0]


streams = pd.read_parquet(find_input("streams_features.parquet"))
wimans = pd.read_parquet(find_input("wimans_clip_features.parquet"))
csib = pd.read_parquet(find_input("csibench_clip_features.parquet"))

sys.path.insert(0, "/kaggle/working")
import esp32_counter as ec

OUT = "/kaggle/working/phase5"
FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)

BASE = list(ec.FEATURE_NAMES)
EXTRA = ["pc2_frac", "pc3_frac", "eig_entropy", "doppler_centroid", "doppler_spread", "psd_entropy",
         "activity_cv", "tone_energy_cv", "half_band_corr"]
MEM = ["mem_cv_mean", "mem_cv_std", "mem_diff_mean", "mem_diff_std"]
LOG_COLS = list(ec.LOG_FEATURES)                    # log() before the offset, as on the device
MODES = ["zero_shot", "empty_cal"]
CAL_COUNT = 2                                       # second point of the two-point calibration
SEED = 0

GBM_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED,
                  verbose=-1, n_jobs=os.cpu_count())

for name, d in [("streams", streams), ("wimans", wimans), ("csibench", csib)]:
    print(f"{name:9s} {d.shape}  datasets={sorted(d['dataset'].unique())}")

In [ ]:
# 2. One table, the 30 s memory cues, links, and the calibration windows
clips = pd.concat([wimans, csib], ignore_index=True)
clips["receiver"] = clips["view"]
clips["window"] = 0
clips["t_start_s"] = 0.0
streams["session"] = streams["recording"]
streams["sample_no"] = 0

cols = ["dataset", "recording", "setup", "receiver", "session", "sample_no", "count", "view", "window",
        "t_start_s"] + BASE + EXTRA
DATA = pd.concat([streams[cols], clips[cols]], ignore_index=True)
DATA["stream"] = DATA["dataset"].isin(["mat80", "wicount", "countfi"])

# ---- 30 s memory: mean / std of the last 12 windows (only the past) per recording + view ----
DATA = DATA.sort_values(["dataset", "recording", "view", "window"]).reset_index(drop=True)
grp = DATA.groupby(["dataset", "recording", "view"])
for src, name in [("cv_median", "cv"), ("diff_energy", "diff")]:
    roll = grp[src].rolling(ec.SMOOTH_N, min_periods=1)
    DATA[f"mem_{name}_mean"] = roll.mean().reset_index(level=[0, 1, 2], drop=True)
    DATA[f"mem_{name}_std"] = roll.std().reset_index(level=[0, 1, 2], drop=True).fillna(0.0)
DATA.loc[~DATA["stream"], MEM] = np.nan             # clips have no past

# ---- links: which rows are 'the same moment seen by different receivers' ----
#   mat80: antennas of one sub-band | WiMANS: receive antennas of one transmit antenna
#   CSI-Bench: the 3 devices | CountFi: the 2 receivers (same scenario, count, sub-band, window index)
v = DATA["view"].astype(str)
DATA["link"] = "0"
DATA["moment"] = DATA["recording"] + "|" + DATA["window"].astype(str)
m = DATA["dataset"] == "mat80"
DATA.loc[m, "link"] = v[m].str.split("_").str[0]
DATA.loc[m, "moment"] = DATA["recording"][m] + "|" + v[m].str.split("_").str[1] + "|" + DATA["window"][m].astype(str)
m = DATA["dataset"] == "wimans"
DATA.loc[m, "link"] = v[m].str.split("_").str[1]
DATA.loc[m, "moment"] = DATA["recording"][m] + "|" + v[m].str.split("_").str[0]
m = DATA["dataset"] == "csibench"
DATA.loc[m, "link"] = v[m]
DATA.loc[m, "moment"] = DATA["recording"][m]
m = DATA["dataset"] == "countfi"
DATA.loc[m, "link"] = DATA["receiver"][m]
DATA.loc[m, "moment"] = (DATA["setup"][m] + "|" + DATA["count"][m].astype(str) + "|" + v[m]
                         + "|" + DATA["window"][m].astype(str))

# ---- calibration windows (never trained on, never scored) ----
#   streams: first 20 s after the warm-up of the first recording with that count in the setup
#   clips:   first 7 clips of the first session with that count in the room
CAL_END = ec.WARMUP_S + ec.CAL_S + 1e-9


def mark_cal(count):
    mask = np.zeros(len(DATA), dtype=bool)
    for (ds, setup), g in DATA[DATA["count"] == count].groupby(["dataset", "setup"]):
        if g["stream"].iloc[0]:
            rec = sorted(g["recording"].unique())[0]
            mask |= ((DATA["dataset"] == ds) & (DATA["recording"] == rec)
                     & (DATA["t_start_s"] + ec.WIN_S <= CAL_END)).values
        else:
            first = sorted(g["session"].unique())[0]
            recs = (g[g["session"] == first].drop_duplicates("recording")
                    .sort_values("sample_no")["recording"].head(ec.N_CAL_WINDOWS))
            mask |= ((DATA["dataset"] == ds) & DATA["recording"].isin(recs)).values
    return mask


DATA["cal"] = ""
DATA.loc[mark_cal(0), "cal"] = "empty"
DATA.loc[mark_cal(CAL_COUNT) & (DATA["cal"] == ""), "cal"] = "two"
DATA["site"] = np.where(DATA["dataset"] == "wicount", DATA["recording"], DATA["setup"])

display(pd.crosstab(DATA["dataset"], DATA["cal"]))
display(DATA.drop_duplicates(["dataset", "recording"]).pivot_table(
    index="dataset", columns="count", values="recording", aggfunc="count", fill_value=0))

In [ ]:
# 3. Model input (log + empty-room offset, as on the device), training, scoring
def prep(F, cols, offset=None):
    T = np.nan_to_num(np.asarray(F, dtype=np.float64), nan=0.0).copy()
    for k, c in enumerate(cols):
        if c in LOG_COLS:
            T[:, k] = np.log(np.maximum(T[:, k], 0.0) + ec.EPS)
    if offset is not None:
        T = T - offset
    return T.astype(np.float32)


def offsets_for(cols):
    """Empty-room offset per (dataset, setup, view) = mean model input over the empty calibration windows."""
    cal = DATA[DATA["cal"] == "empty"]
    return {key: prep(g[cols].values, cols).astype(np.float64).mean(axis=0)
            for key, g in cal.groupby(["dataset", "setup", "view"])}


OFFSETS = {}


def model_inputs(df, cols, mode):
    df = df.reset_index(drop=True)
    if mode == "zero_shot":
        return prep(df[cols].values, cols), np.ones(len(df), dtype=bool)
    key_cols = tuple(cols)
    if key_cols not in OFFSETS:
        OFFSETS[key_cols] = offsets_for(cols)
    off = OFFSETS[key_cols]
    X = np.zeros((len(df), len(cols)), dtype=np.float32)
    ok = np.zeros(len(df), dtype=bool)
    for key, idx in df.groupby(["dataset", "setup", "view"]).indices.items():
        if key in off:
            X[idx] = prep(df[cols].values[idx], cols, off[key])
            ok[idx] = True
    return X, ok


def fit(train_df, cols, mode, task):
    X, ok = model_inputs(train_df, cols, mode)
    d = train_df.reset_index(drop=True)[ok]
    w = 1.0 / d.groupby("dataset")["dataset"].transform("size").values     # each dataset the same weight
    w = w / w.mean()
    if task == "clf4":
        model = lgb.LGBMClassifier(objective="multiclass", **GBM_PARAMS)
        model.fit(X[ok], np.minimum(d["count"].values, 3), sample_weight=w)
        return model, 3
    model = lgb.LGBMRegressor(objective="regression", **GBM_PARAMS)
    model.fit(X[ok], d["count"].values, sample_weight=w)
    return model, int(d["count"].max())


def predict(model, max_count, df, cols, mode):
    X, ok = model_inputs(df, cols, mode)
    p = np.full(len(df), np.nan)
    if ok.any():
        p[ok] = np.clip(model.predict(X[ok]), 0, max_count)
    return p


def f1_macro(y, r):
    labels = sorted(set(y) | set(r))
    out = []
    for c in labels:
        tp = np.sum((y == c) & (r == c))
        out.append(2 * tp / max(2 * tp + np.sum((y != c) & (r == c)) + np.sum((y == c) & (r != c)), 1))
    return float(np.mean(out))


def score(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    r = np.rint(pred)
    y4, r4 = np.minimum(y, 3), np.minimum(r, 3)
    mae_const = float(np.mean(np.abs(y - y.mean())))
    mae = float(np.mean(np.abs(pred - y)))
    return {"n": len(y), "mae": mae, "rel_mae": mae / max(mae_const, 1e-9),
            "exact": float(np.mean(r == y)), "within1": float(np.mean(np.abs(r - y) <= 1)),
            "acc4": float(np.mean(r4 == y4)), "f1_4": f1_macro(y4, r4),
            "empty_recall": float(np.mean(r[y == 0] == 0)) if (y == 0).any() else np.nan}

In [ ]:
# 4. Leave one site out for every variant and mode (predictions also on each site's calibration windows)
POOL_ALL = ["wicount", "mat80", "wimans", "csibench"]          # = Phase 4 final pool
POOL_STREAMS = ["wicount", "mat80"]                            # the memory cues exist only for streams
VARIANTS = {
    "base_reg":         dict(cols=BASE, task="reg", pool=POOL_ALL),            # = Phase 4 final model
    "extra_reg":        dict(cols=BASE + EXTRA, task="reg", pool=POOL_ALL),
    "base_clf4":        dict(cols=BASE, task="clf4", pool=POOL_ALL),
    "base_reg_streams": dict(cols=BASE, task="reg", pool=POOL_STREAMS),        # reference for mem_reg
    "mem_reg":          dict(cols=BASE + MEM, task="reg", pool=POOL_STREAMS),
}
ALL_DS = ["mat80", "wicount", "wimans", "csibench", "countfi"]

TRAIN = DATA[DATA["cal"] == ""]
CAL = DATA[DATA["cal"] != ""]
KEEP = ["dataset", "recording", "setup", "site", "receiver", "count", "view", "window", "link", "moment", "cal"]
parts = []
t0 = time.time()

for vname, cfg in VARIANTS.items():
    cols, task, pool = cfg["cols"], cfg["task"], cfg["pool"]
    test_ds = ALL_DS if vname not in ("mem_reg", "base_reg_streams") else ["mat80", "wicount", "countfi"]
    pool_df = TRAIN[TRAIN["dataset"].isin(pool)]

    for mode in MODES:
        full = fit(pool_df, cols, mode, task)
        sites = (TRAIN[TRAIN["dataset"].isin(test_ds)][["dataset", "site"]]
                 .drop_duplicates().values.tolist())

        for ds, site in sites:
            if ds in pool:
                held = (pool_df["dataset"] == ds) & (pool_df["site"] == site)
                model, mc = fit(pool_df[~held], cols, mode, task)
            else:
                model, mc = full
            test = TRAIN[(TRAIN["dataset"] == ds) & (TRAIN["site"] == site)]
            setups = test["setup"].unique()
            cal = CAL[(CAL["dataset"] == ds) & CAL["setup"].isin(setups)]
            both = pd.concat([test, cal])
            d = both[KEEP].copy()
            d["raw"] = predict(model, mc, both, cols, mode)
            d["fold"] = site
            d["variant"], d["mode"], d["max_count"] = vname, mode, mc
            parts.append(d[~d["raw"].isna()])

        print(f"{vname:17s} {mode:9s} {len(sites):3d} sites  [{time.time() - t0:.0f} s]", flush=True)

P = pd.concat(parts, ignore_index=True)
P = P.sort_values(["variant", "mode", "fold", "recording", "view", "window"]).reset_index(drop=True)

# ---- the display value: causal median of the last 30 s per stream (clips: one estimate each) ----
P["smoothed"] = P.groupby(["variant", "mode", "fold", "recording", "view"])["raw"].transform(
    lambda s: ec.smooth_predictions(s.values))
print(len(P), "predictions")

In [ ]:
# 5. Two-point calibration: empty -> 0 and the 2-people calibration -> 2 (per fold, site and link)
cal_pts = (P[P["cal"] != ""].groupby(["variant", "mode", "fold", "dataset", "setup", "view", "cal"])["smoothed"]
           .median().unstack("cal"))
cal_pts = cal_pts.rename(columns={"empty": "p0", "two": "p2"}).reset_index()
cal_pts = cal_pts.dropna(subset=["p0", "p2"])
cal_pts = cal_pts[cal_pts["p2"] - cal_pts["p0"] > 0.2]          # flat response -> no stretch possible

S = P[P["cal"] == ""].merge(cal_pts, on=["variant", "mode", "fold", "dataset", "setup", "view"], how="left")
S["two_point"] = np.clip((S["smoothed"] - S["p0"]) * CAL_COUNT / (S["p2"] - S["p0"]), 0, S["max_count"])
S.loc[S["variant"] == "base_clf4", "two_point"] = np.nan        # a class cannot be stretched
print("rows with a two-point calibration:", int(S["two_point"].notna().sum()), "/", len(S))
display(cal_pts.groupby(["variant", "mode", "dataset"]).size().rename("calibrated links").unstack(0))

In [ ]:
# 6. Score: single link and 2 / 3 links averaged, without / with the two-point calibration
MAX_COMBOS = 12
rows = []


def link_combos(links, k):
    combos = list(itertools.combinations(sorted(links), k))
    return combos[:MAX_COMBOS]


for (vname, mode, ds), g in S.groupby(["variant", "mode", "dataset"]):
    links = g["link"].unique()
    for value in ["smoothed", "two_point"]:
        h0 = g[g[value].notna()]
        if h0.empty:
            continue
        # k links: average over every k-link combination, only moments where all k links have a value
        for k in [1, 2, 3]:
            if k > len(links):
                continue
            ys, ps = [], []
            for combo in link_combos(links, k):
                h = h0[h0["link"].isin(combo)]
                agg = h.groupby("moment").agg(y=("count", "first"), p=(value, "mean"), n=("link", "nunique"))
                agg = agg[agg["n"] == k]
                ys.append(agg["y"].values)
                ps.append(agg["p"].values)
            if not ys or not len(np.concatenate(ys)):
                continue
            rows.append({"variant": vname, "mode": mode, "dataset": ds, "calibration": value,
                         "links": k, **score(np.concatenate(ys), np.concatenate(ps))})

R = pd.DataFrame(rows)
R["calibration"] = R["calibration"].map({"smoothed": "none / empty only", "two_point": "two-point"})
R.to_csv(os.path.join(OUT, "all_scores.csv"), index=False)
show = ["variant", "mode", "dataset", "calibration", "links", "n", "mae", "rel_mae", "exact", "within1",
        "acc4", "f1_4", "empty_recall"]
R[R["links"] == 1][show].round(3)

In [ ]:
# 7. Each upgrade against its baseline, on exactly the same windows
def paired(rows, value_a, value_b, by, cap3=False):
    """Score two prediction columns on the rows where both exist (cap3: on the 0/1/2/3+ scale)."""
    out = []
    for key, g in rows.groupby(by):
        h = g.dropna(subset=[value_a, value_b])
        if h.empty:
            continue
        y, a, b = h["count"], h[value_a], h[value_b]
        if cap3:
            y, a, b = np.minimum(y, 3), np.minimum(a, 3), np.minimum(b, 3)
        sa, sb = score(y, a), score(y, b)
        out.append({**dict(zip(by, key)), "n": len(h),
                    "mae_base": sa["mae"], "mae_new": sb["mae"], "d_mae": sb["mae"] - sa["mae"],
                    "f1_4_base": sa["f1_4"], "f1_4_new": sb["f1_4"], "d_f1_4": sb["f1_4"] - sa["f1_4"]})
    return pd.DataFrame(out)


single = S[S["cal"] == ""]
key = ["mode", "fold", "dataset", "recording", "view", "window"]
wide = single.pivot_table(index=key + ["count"], columns="variant", values="smoothed").reset_index()

comparisons = []
for new, base in [("extra_reg", "base_reg"), ("base_clf4", "base_reg"), ("mem_reg", "base_reg_streams")]:
    if new in wide and base in wide:
        cap3 = new == "base_clf4"                  # the classifier only knows 0 / 1 / 2 / 3+
        c = paired(wide, base, new, ["mode", "dataset"], cap3=cap3)
        c.insert(0, "upgrade", f"{new} vs {base}" + (" (0/1/2/3+ scale)" if cap3 else ""))
        comparisons.append(c)

# two-point vs none, base model
tp = S[(S["variant"] == "base_reg")]
c = paired(tp, "smoothed", "two_point", ["mode", "dataset"])
c.insert(0, "upgrade", "two-point calibration vs none")
comparisons.append(c)

# 2 / 3 links vs 1 link, base model (from the link table)
lk = R[(R["variant"] == "base_reg")].pivot_table(index=["mode", "dataset", "calibration"], columns="links",
                                                  values="mae").reset_index()
for k in [2, 3]:
    if k in lk:
        c = lk[["mode", "dataset", "calibration", 1, k]].dropna().rename(columns={1: "mae_base", k: "mae_new"})
        c["d_mae"] = c["mae_new"] - c["mae_base"]
        c.insert(0, "upgrade", f"{k} links vs 1 link (base_reg)")
        comparisons.append(c)

CMP = pd.concat(comparisons, ignore_index=True)
CMP.to_csv(os.path.join(OUT, "upgrades_vs_baseline.csv"), index=False)
CMP.round(3)

In [ ]:
# 8. Figures: error vs number of links; mean estimate per true count with / without two-point
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, cal in zip(axes, ["none / empty only", "two-point"]):
    h = R[(R["variant"] == "base_reg") & (R["mode"] == "empty_cal") & (R["calibration"] == cal)]
    for ds, g in h.groupby("dataset"):
        ax.plot(g["links"], g["mae"], "o-", label=ds)
    ax.set_xticks([1, 2, 3])
    ax.set_xlabel("links averaged")
    ax.set_ylabel("MAE (people)")
    ax.set_title(f"base model, empty_cal, calibration: {cal}")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "mae_vs_links.png"), dpi=120)
plt.show()

b = S[(S["variant"] == "base_reg") & (S["mode"] == "empty_cal")]
per_count = (b.groupby(["dataset", "count"])
             .agg(n=("smoothed", "size"), mean_none=("smoothed", "mean"), mean_two_point=("two_point", "mean"))
             .reset_index())
per_count.to_csv(os.path.join(OUT, "per_count_two_point.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, col, title in zip(axes, ["mean_none", "mean_two_point"], ["empty room only", "two-point"]):
    for ds, g in per_count.groupby("dataset"):
        ax.plot(g["count"], g[col], "o-", label=ds)
    top = int(per_count["count"].max())
    ax.plot([0, top], [0, top], "k--", lw=0.8)
    ax.set_xlabel("true people")
    ax.set_title(f"mean estimate, one link - {title}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("mean estimate")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "per_count_two_point.png"), dpi=120)
plt.show()
per_count.round(2)

In [ ]:
# 9. Verdicts + PHASE5_RESULTS.md
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:                              # tabulate missing
        return "```" + chr(10) + df.to_string(index=False) + chr(10) + "```"


# An upgrade 'helps' when it lowers the MAE by >= 0.05 people on at least 2 datasets
# and does not make the ESP32 data (WiCount) worse by more than 0.05.
verdicts = []
for up, g in CMP.groupby("upgrade"):
    for mode, h in g.groupby("mode"):
        if "calibration" in h and h["calibration"].notna().any():
            h = h[h["calibration"] == "none / empty only"]
        better = h.loc[h["d_mae"] <= -0.05, "dataset"].tolist()
        worse = h.loc[h["d_mae"] >= 0.05, "dataset"].tolist()
        wic = h.loc[h["dataset"] == "wicount", "d_mae"]
        wic_ok = wic.empty or float(wic.iloc[0]) < 0.05
        helps = len(better) >= 2 and wic_ok
        verdicts.append({"upgrade": up, "mode": mode, "helps": helps, "better_on": ", ".join(better),
                         "worse_on": ", ".join(worse),
                         "wicount_d_mae": None if wic.empty else round(float(wic.iloc[0]), 3)})
V = pd.DataFrame(verdicts)
V.to_csv(os.path.join(OUT, "verdicts.csv"), index=False)
display(V)

base1 = R[(R["variant"] == "base_reg") & (R["links"] == 1)]
lines = [
    "# Phase 5 results - which upgrades make the ESP32 counter better", "",
    f"Generated {time.strftime('%Y-%m-%d %H:%M')} on Kaggle. Leave one site out on all 5 datasets; "
    "stream datasets scored after the 30 s median, clips one estimate each.", "",
    "Rule: an upgrade **helps** when it lowers the MAE by >= 0.05 people on at least 2 datasets and does not make "
    "WiCount (ESP32) worse by more than 0.05.", "",
    "## Verdicts", "", md(V), "",
    "## Baseline = Phase 4 model (base_reg), one link", "",
    md(base1[show].round(3)), "",
    "## Every upgrade against its baseline (same windows)", "",
    md(CMP.round(3)), "",
    "## Links: 1 / 2 / 3 averaged (base_reg)", "",
    md(R[R["variant"] == "base_reg"][show].round(3)), "",
    "## Mean estimate per true count, empty_cal, one link: empty room only vs two-point", "",
    md(per_count.round(2)), "",
    "## All variants, one link", "",
    md(R[R["links"] == 1][show].round(3)), "",
    "## Notes", "",
    "- Links: mat80 = antennas of one router, WiMANS = receive antennas of one card (both close together, so a lower "
    "bound for 2 boards on different walls); CSI-Bench = 3 devices; CountFi = 2 receivers matched by window index.",
    "- Two-point calibration uses 20 s of the empty room and 20 s with 2 people of the same site; those windows are "
    "never trained on or scored. Not possible on CountFi (no empty) or CSI-Bench (no empty).",
    "- CountFi is never trained on; each count there is one time block (stress test only).",
    "- WiMANS / CSI-Bench clips are clip-mean normalised; they have no 30 s memory, so mem_reg is streams only.",
    "- No model is exported: each upgrade needs esp32_counter.py / firmware changes first.",
]
with open(os.path.join(OUT, "PHASE5_RESULTS.md"), "w") as f:
    f.write(chr(10).join(lines))
print("saved", os.path.join(OUT, "PHASE5_RESULTS.md"))
for name in sorted(os.listdir(OUT)):
    print(" ", name)